In [ ]:
# ============================================================
# Exercise 3: Algorithmus für Cadence & Aktivitätsklassifikation
# ============================================================

from DataProcessor import DataProcessor
from scipy.signal import butter, filtfilt, welch
import numpy as np

# ------------------------------------------------------------
# 1. Hilfsfunktionen (Filter, Magnitude, Cadence, dominante Frequenz)
# ------------------------------------------------------------
def bandpass_filter(signal, fs, lowcut=0.5, highcut=5.0, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

def total_acc_magnitude(x, y, z):
    """Gesamtbeschleunigung (Magnitude) minus Mittelwert (Gravitation)"""
    mag = np.sqrt(x**2 + y**2 + z**2)
    return mag - np.mean(mag)

def estimate_cadence(signal, fs):
    """
    Cadence in steps/min aus Autokorrelation (positive Lags).
    signal: 1D-Array (Magnitude der Beschleunigung)
    """
    corr = np.correlate(signal, signal, mode='full')
    corr = corr / len(signal)                # biased
    pos_corr = corr[len(signal):]            # nur positive Lags
    
    # Suche Schrittfrequenz zwischen 0.8 Hz und 3.5 Hz
    min_lag = int(fs / 3.5)
    max_lag = int(fs / 0.8)
    max_lag = min(max_lag, len(pos_corr))
    
    if min_lag >= len(pos_corr) or max_lag <= min_lag:
        return 0.0
    
    peak_idx = np.argmax(pos_corr[min_lag:max_lag])
    peak_lag = min_lag + peak_idx
    return (60.0 / (peak_lag / fs))

def dominant_frequency(signal, fs):
    """Dominante Frequenz (Hz) aus Welch‑Periodogramm"""
    sig_filt = bandpass_filter(signal, fs)
    nperseg = min(2*fs, len(sig_filt))
    freqs, Pxx = welch(sig_filt, fs=fs, nperseg=nperseg, noverlap=nperseg//2)
    mask = (freqs >= 0.5) & (freqs <= 3.5)
    if not np.any(mask):
        return 0.0
    idx = np.argmax(Pxx[mask])
    return freqs[mask][idx]

def classify_activity(cadence, dom_freq):
    """Schwellwertlogik (Reihenfolge: Rennen → schnell → normal → ruhen)"""
    if dom_freq >= 2.0 or cadence > 155:
        return 4          # Rennen
    elif 1.4 <= dom_freq < 2.2 and 100 <= cadence <= 160:
        return 3          # Schnell Gehen
    elif 1.0 <= dom_freq < 1.6 and 60 <= cadence <= 115:
        return 2          # Normal Gehen
    elif cadence < 30 and dom_freq < 0.5:
        return 1          # Ruhen
    else:
        return 0          # Nicht klassifizierbar

# ------------------------------------------------------------
# 2. Hauptteil: Daten laden, in Blöcke teilen, Klassifikation
# ------------------------------------------------------------
BLOCK_DURATION = 5.0      # Sekunden
ACTIVITY_NAMES = {0:"unbekannt", 1:"Ruhen", 2:"Gehen", 3:"Schnell Gehen", 4:"Rennen"}

# Pfade (passe an deine tatsächlichen Dateien an)
dp = DataProcessor("rawdata/X22/")
files = ['rawdata/X22/normal_gehen3.pickle', 'rawdata/X22/schnell_Laufen10.pickle', 'rawdata/X22/rennen1.pickle'];

for i in range(3): 
    fileName = files[i]
    print(f"\n--- {fileName} ---")
    # Daten laden (genau wie in Exercise 1)
    dp.loadRawData(fileName)
    for dev in dp.getDevices():
        dp.loadRawDataDevice(dev)
    
    # Beschleunigungsdaten aus dem DataFrame holen
    x = dp.dfAcc["x"].values
    y = dp.dfAcc["y"].values
    z = dp.dfAcc["z"].values
    fs = dp.fs
    
    # (Optional) Bewegungsbeginn finden – wie in deiner Funktion find_activity_start
    # Hier lassen wir es einfach und nehmen das ganze Signal.
    
    # Blockweise Verarbeitung
    block_samples = int(BLOCK_DURATION * fs)
    num_blocks = len(x) // block_samples
    
    if num_blocks == 0:
        print(f"Signal zu kurz – nur {len(x)} samples (< {block_samples})")
        continue
    
    cadences = []
    activities = []
    
    for b in range(num_blocks):
        start = b * block_samples
        end = start + block_samples
        
        # Magnitude für diesen Block
        mag = total_acc_magnitude(x[start:end], y[start:end], z[start:end])
        
        cad = estimate_cadence(mag, fs)
        dom = dominant_frequency(mag, fs)
        act = classify_activity(cad, dom)
        
        cadences.append(cad)
        activities.append(act)
    
    # Ausgabe
    print(f"Samplingrate: {fs} Hz")
    print(f"Anzahl 5‑Sekunden‑Blöcke: {len(cadences)}")
    if len(cadences) > 0:
        print(f"Cadences (steps/min): {[f'{c:.1f}' for c in cadences]}")
        print(f"Aktivitäten (Code/Name): {[(a, ACTIVITY_NAMES[a]) for a in activities]}")
        print(f"Mittlere Cadence: {np.mean(cadences):.1f} steps/min")
    else:
        print("Keine vollständigen Blöcke.")